In [ ]:
!pip install transformers==4.51.2 sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.7 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.0
    Uninstalling tokenizers-0.22.0:
      Successfully uninstalled tokenizers-0.22.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.1
    Uninstalling transformers-4.56.1:
      Successfully uninstalled transformers-4.56.1


# Imports

In [ ]:
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import os
import torch

# Load data

In [ ]:
from google.colab import drive
if not os.path.exists('/gd'):
    drive.mount('/gd')

Mounted at /gd


In [ ]:
# Load the revised dataset
data_revised = pd.read_csv('/gd/MyDrive/NLLB Data/AWAL evaluation sets - FLORES devtest (REVISED).csv')

data_revised["ID"] = data_revised.index

# remove unnecessary columns
data_revised = data_revised[["ID", "English", "Tamazight (Corrected)"]]
data_revised.columns = ["ID", "eng", "zgh"]

data_revised.describe(include='all')

,ID,eng,zgh
count,1012.000000,1012,1012
unique,NaN,1012,1012
top,NaN,Workers must often get their superiors' approv...,ⵉⴳⴳⵓⴷⵉ ⵎⴰ ⴳ ⴷ ⵉⵇⵇⴰⵏ ⴰⴷ ⵢⵉⵍⵉ ⵓⵎⵙⴰⵙⴰ ⵏ ⵉⵏⵙⵙⵉⵅⴼⵏ ...
freq,NaN,1,1
mean,505.500000,NaN,NaN
std,292.283538,NaN,NaN
min,0.000000,NaN,NaN
25%,252.750000,NaN,NaN
50%,505.500000,NaN,NaN
75%,758.250000,NaN,NaN


In [ ]:
# Load the original dataset
eng_data_original = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
zgh_data_original = load_dataset("openlanguagedata/flores_plus", "zgh_Tfng", split="devtest")

data_original = pd.DataFrame({
    "ID": eng_data_original["id"],
    "eng": eng_data_original["text"],
    "zgh": zgh_data_original["text"],
})

data_original.describe(include='all')

README.md:   0%|          | 0.00/72.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

dev/eng_Latn.parquet:   0%|          | 0.00/112k [00:00<?, ?B/s]

devtest/eng_Latn.parquet:   0%|          | 0.00/117k [00:00<?, ?B/s]

...

dev/zgh_Tfng.parquet:   0%|          | 0.00/154k [00:00<?, ?B/s]

devtest/zgh_Tfng.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

,ID,eng,zgh
count,1012.000000,1012,1012
unique,NaN,1012,1012
top,NaN,Workers must often get their superiors' approv...,ⵉⴳⴳⵓⵜ ⵎⴰⴳ ⴷ ⵉⵇⵇⴰⵏ ⴰⴷ ⵢⵉⵍⵉ ⵓⵎⵙⴰⵙⴰ ⵏ ⵉⵏⵙⵙⵉⵅⴼⵏ ⵏ ...
freq,NaN,1,1
mean,505.500000,NaN,NaN
std,292.283538,NaN,NaN
min,0.000000,NaN,NaN
25%,252.750000,NaN,NaN
50%,505.500000,NaN,NaN
75%,758.250000,NaN,NaN


In [ ]:
# sanity check
assert data_original['eng'].equals(data_revised['eng']), "English texts do not match!"
assert not data_original['zgh'].equals(data_revised['zgh']), "Revised and original Tamazight texts should not match!"

# Preprocess data

In [ ]:
# replace the 'ⵒ', 'ⵁ', and 'ⴴ' letters with 'ⴱ', 'ⵀ', and 'ⵖ' respectively, since they're not in the model's vocabulary
data_revised['zgh'] = data_revised['zgh'].str.replace('ⵒ', 'ⴱ')
data_revised['zgh'] = data_revised['zgh'].str.replace('ⵁ', 'ⵀ')
data_revised['zgh'] = data_revised['zgh'].str.replace('ⴴ', 'ⵖ')

data_original['zgh'] = data_original['zgh'].str.replace('ⵒ', 'ⴱ')
data_original['zgh'] = data_original['zgh'].str.replace('ⵁ', 'ⵀ')
data_original['zgh'] = data_original['zgh'].str.replace('ⴴ', 'ⵖ')

In [ ]:
# this code is adapted from  the Stopes repo of the NLLB team
# https://github.com/facebookresearch/stopes/blob/main/stopes/pipelines/monolingual/monolingual_line_processor.py#L214

import re
import sys
import typing as tp
import unicodedata
from sacremoses import MosesPunctNormalizer


mpn = MosesPunctNormalizer(lang="en")
mpn.substitutions = [
    (re.compile(r), sub) for r, sub in mpn.substitutions
]


def get_non_printing_char_replacer(replace_by: str = " ") -> tp.Callable[[str], str]:
    non_printable_map = {
        ord(c): replace_by
        for c in (chr(i) for i in range(sys.maxunicode + 1))
        # same as \p{C} in perl
        # see https://www.unicode.org/reports/tr44/#General_Category_Values
        if unicodedata.category(c) in {"C", "Cc", "Cf", "Cs", "Co", "Cn"}
    }

    def replace_non_printing_char(line) -> str:
        return line.translate(non_printable_map)

    return replace_non_printing_char

replace_nonprint = get_non_printing_char_replacer(" ")

def preproc(text):
    clean = mpn.normalize(text)
    clean = replace_nonprint(clean)
    # replace 𝓕𝔯𝔞𝔫𝔠𝔢𝔰𝔠𝔞 by Francesca
    clean = unicodedata.normalize("NFKC", clean)
    return clean

In [ ]:
# apply preprocessing
data_revised['eng'] = data_revised['eng'].apply(preproc)
data_revised['zgh'] = data_revised['zgh'].apply(preproc)

data_original['eng'] = data_original['eng'].apply(preproc)
data_original['zgh'] = data_original['zgh'].apply(preproc)

# Evaluation

In [ ]:
NLLB_LANG_CODES = {
    "zgh": "tzm_Tfng",
    "eng": "eng_Latn"
}

In [ ]:
MAX_LENGTH = 238
NUM_BEAMS = 8

In [ ]:
MODEL_ID = "Tamazight-NLP/NLLB-200-600M-Tamazight-All-Data-1.25-epoch"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=True, src_lang="eng_Latn"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, token=True).to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/842 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

In [ ]:
def get_nllb_translations(data, tokenizer, model, source_lang_code, target_lang_code, **kwargs):
    translations = []

    tokenizer.src_lang = NLLB_LANG_CODES[source_lang_code]

    for _, row in tqdm(data.iterrows(), total=len(data)):
        id = row["ID"]
        text = row[source_lang_code]

        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(NLLB_LANG_CODES[target_lang_code]),
            **kwargs
            )
        translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

        translations.append({
            "ID": id,
            "prediction": translation
        })

    return pd.DataFrame(translations)

# Revised data ENG->ZGH

In [ ]:
eng_zgh_translations = get_nllb_translations(data_revised, tokenizer, model, "eng", "zgh", max_length=MAX_LENGTH, num_beams=NUM_BEAMS)
eng_zgh_translations

100%|██████████| 1012/1012 [26:43<00:00,  1.58s/it]


,ID,prediction
0,0,"""ⵉⵍⴰ ⴷⵖⵉ ⵖⵓⵔⵏⵖ ⵉⴼⵔⴷⵉⵙⵏ ⵉⵍⴰⵏ 4 ⵏ ⵡⴰⵢⵢⵓⵔⵏ ⵏⵏⴰ ⵓⵔ..."
1,1,"ⵢⵓⵡⴹ ⴷⵓⴽⵜⵓⵔ ⵉⴷ ⵓⵕ, ⴰⵙⵍⵎⴰⴷ ⵏ ⵜⵎⴰⵙⵙⴰⵏⵜ ⵏ ⵓⵙⴳⵏⴰⴼ ..."
2,2,"ⵣⵓⵏ ⴷ ⴽⵔⴰ ⵏ ⵉⵎⵓⵙⵏⴰⵡⵏ ⵢⴰⴹⵏ, ⵓⵔ ⵉⵙⵙⵉⵏ ⵉⵙ ⵉⵖⵉⵢ ⵓⵙ..."
3,3,"ⴰⵙⵙ ⵏ ⵓⵢⵏⴰⵙ, ⵜⵏⵏⴰ ⵙⴰⵔⴰ ⴷⴰⵏⵢⵓⵙ, ⵜⴰⵎⵙⵙⵓⴳⵓⵔⵜ ⵜⴰⵎⴷ..."
4,4,"ⵉⵏⵏⴰ ⴷⴰⵏⵢⵓⵙ, ""ⴳ ⵜⵉⵣⵉ ⴰⴷ ⵓⵔ ⴷⴰ ⵏⵙⴽⴰⵔ ⴰⵎⵢⴰ. ⵖⵔⵉⵖ..."
...,...,...
1007,1007,"ⴰⵛⴽⵓ ⴳⴰⵏ ⵡⴰⵏⵙⵉⵡⵏ ⵉⵎⵣⴷⴰⵖ ⵉⴷⵔⵓⵙⵏ, ⴷ ⵓⵙⵅⵙⵉ ⵏ ⵓⵙⵉⴷ..."
1008,1008,ⵜⴰⴷⵍⵙⴰ ⵏ ⵜⵡⵓⵔⵉ ⵜⴰⵢⴰⴱⴰⵏⵉⵜ ⵜⴳⴰ ⵜⴰⴷⵍⵙⴰ ⵜⴰⵣⵣⵓⵍⴰⵏⵜ ...
1009,1009,"ⴳⴰⵏⵜ ⵜⵎⵍⵙⴰ ⵜⵉⴽⵙⵡⴰⵜⵉⵏ ⵜⵉⴽⵙⵡⴰⵜⵉⵏ ⵏ ⵜⵡⵓⵔⵉⵡⵉⵏ, ⴰⵔ ..."
1010,1010,"ⵉⴳⴰ ⵓⵎⵙⴰⵙⴰ ⴳ ⵡⴰⵏⵙⵉⵡⵏ ⵏ ⵜⵡⵓⵔⵉ ⴰⵅⴰⵜⴰⵔ, ⴰⵛⴽⵓ ⴷⴰ ⵉ..."


In [ ]:
# save the answers to a CSV file
eng_zgh_translations.to_csv(f"nllb-600m-all-data-1_25-epoch-{NUM_BEAMS}-beam-eng-zgh-revised.csv", index=False)

# Revised data ZGH->ENG

In [ ]:
zgh_eng_translations = get_nllb_translations(data_revised, tokenizer, model, "zgh", "eng", max_length=MAX_LENGTH, num_beams=NUM_BEAMS)
zgh_eng_translations

100%|██████████| 1012/1012 [12:57<00:00,  1.30it/s]


,ID,prediction
0,0,"""We now have 4 month old rats not infected wit..."
1,1,"Dr. Ehud O'Neill, a professor of medicine at D..."
2,2,"Like other hypotheses, it is complicated by th..."
3,3,"In January, Sara Danius, secretary general of ..."
4,4,"""We're not doing anything right now. I called ..."
...,...,...
1007,1007,"If the area is sparsely populated, and a few v..."
1008,1008,"Japanese working, performing and performing cu..."
1009,1009,"Costumes are the official attire of employers,..."
1010,1010,Consistency is more important in the workplace...


In [ ]:
# save the answers to a CSV file
zgh_eng_translations.to_csv(f"nllb-600m-all-data-1_25-epoch-{NUM_BEAMS}-beam-zgh-eng-revised.csv", index=False)

# Original data ZGH->ENG

In [ ]:
original_zgh_eng_translations = get_nllb_translations(data_original, tokenizer, model, "zgh", "eng", max_length=MAX_LENGTH, num_beams=NUM_BEAMS)
original_zgh_eng_translations

100%|██████████| 1012/1012 [12:57<00:00,  1.30it/s]


,ID,prediction
0,0,"""We now have 4-year-old rats with no diabetes ..."
1,1,"Dr. Ihor Orr, a professor of medicine at Dalho..."
2,2,"Like many other authors, he was complicit in t..."
3,3,"In January, Sara Danius, secretary-general of ..."
4,4,"""We're not doing anything right now. I called ..."
...,...,...
1007,1007,"If the area is sparsely populated, and a few v..."
1008,1008,Japanese labour and labour culture is more for...
1009,1009,"Costumes are the official attire of employers,..."
1010,1010,Consistency is more important in the workplace...


In [ ]:
# save the answers to a CSV file
original_zgh_eng_translations.to_csv(f"nllb-600m-all-data-1_25-epoch-{NUM_BEAMS}-beam-zgh-eng-original.csv", index=False)